<a href="https://colab.research.google.com/github/boradenihal15-beep/FlyRank_Ai_ML/blob/main/Works/Notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip -q install duckdb huggingface_hub


In [8]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [10]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis

The unit of analysis is a single content item (web page) aggregated from Google Search Console search performance data. Each row in the modeling dataset represents one content item with features calculated from historical search performance.

## Time Window

The data comes from the FlyRank Search Intelligence warehouse and contains historical daily search performance. The available reporting window is verified below using SQL rather than assumed.

In [11]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2025-01-27,2026-06-30,78835655


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Category | Fields | Reason |
|----------|--------|--------|
| Features | imp_prev30, visible_queries, rare_share, anon_share, top_query_share | Used by the model to predict future performance. |
| Label / Proxy | is_declining | Indicates whether impressions declined by more than 20% compared to the previous period. |
| Context | client_hash_id, content_hash_id | Used to identify groups and avoid leakage during validation. Not used as predictive features. |
| Excluded | report_date, future performance values, raw identifiers | Excluded because they can introduce leakage or are not meaningful predictive features. |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# 1. Verify dataset grain
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM {TABLES['fact_daily']}
""").df()

# 2. Verify missing values
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(gsc_impressions) AS impressions_present,
    COUNT(gsc_clicks) AS clicks_present,
    COUNT(gsc_avg_position) AS position_present
FROM {TABLES['fact_daily']}
""").df()

# 3. Verify reporting window
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
""").df()

# 4. Verify number of clients
con.sql(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS clients
FROM {TABLES['dim_clients']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients
0,104


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- The dataset contains historical search performance only and cannot explain why rankings changed.
- Client histories are not balanced; some clients have longer reporting periods than others.
- Early records may contain only Google Search Console data before Google Analytics data becomes available.
- The analysis is observational and supports decision-making; it does not prove causation.
- No client names, URLs, or private search queries are included in the public dataset.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.